In [3]:
import os
import urllib.request

import torch

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
PATH = "input.txt"


def load_text(url=URL, path=PATH):
    """Download the text once (if missing) and return it as one string."""
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def build_vocab(text):
    """Return the sorted list of distinct characters and the two lookup dicts."""
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}   # character -> ID
    itos = {i: ch for i, ch in enumerate(chars)}   # ID -> character
    return chars, stoi, itos


def encode(s, stoi):
    """Text -> list of IDs."""
    return [stoi[ch] for ch in s]


def decode(ids, itos):
    """List of IDs -> text."""
    return "".join(itos[i] for i in ids)


def make_splits(text, stoi, train_frac=0.9):
    """Encode the whole text into one tensor and split it into train / val."""
    data = torch.tensor(encode(text, stoi), dtype=torch.long)
    n_train = int(train_frac * len(data))
    return data, data[:n_train], data[n_train:]


text = load_text()
chars, stoi, itos = build_vocab(text)
data, train_data, val_data = make_splits(text, stoi)

print("characters:", len(text))
print("vocab size:", len(chars))
print(data.shape, train_data.shape, val_data.shape)
print(decode(data[:40].tolist(), itos))


characters: 1115394
vocab size: 65
torch.Size([1115394]) torch.Size([1003854]) torch.Size([111540])
First Citizen:
Before we proceed any fur


In [5]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [6]:
train_data[:9]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [7]:
def get_batch(data, block_size, batch_size):
    """Return a random batch: x (chunks) and y (the same chunks shifted by one char)."""
    starts = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in starts])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in starts])
    return x, y


xb, yb = get_batch(train_data, block_size=8, batch_size=4)
print(xb.shape, yb.shape)
print(repr(decode(xb[0].tolist(), itos)))
print(repr(decode(yb[0].tolist(), itos)))

torch.Size([4, 8]) torch.Size([4, 8])
" stay'd "
"stay'd b"


In [8]:
import torch.nn as nn
import torch.nn.functional as F


class BigramModel(nn.Module):
    """Predict the next char from the current char only (a lookup table)."""

    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, y=None):
        logits = self.table(x)                       # (batch, block, vocab)
        if y is None:
            return logits, None
        b, t, v = logits.shape
        loss = F.cross_entropy(logits.view(b * t, v), y.view(b * t))
        return logits, loss


def train_bigram(model, train_data, steps, block_size, batch_size, lr=1e-2):
    """Train the model for a number of steps; print the loss every 500 steps."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        xb, yb = get_batch(train_data, block_size, batch_size)
        _, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 500 == 0:
            print(step, loss.item())
    return model


@torch.no_grad()
def generate(model, max_new_tokens, start_id=0):
    """Generate text one char at a time by sampling from the model's probabilities."""
    idx = torch.tensor([[start_id]], dtype=torch.long)
    for _ in range(max_new_tokens):
        logits, _ = model(idx)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx[0].tolist()


vocab_size = len(chars)
model = BigramModel(vocab_size)
xb, yb = get_batch(train_data, block_size=8, batch_size=4)
_, loss = model(xb, yb)
print("vocab size:", vocab_size, "| start loss:", loss.item())

vocab size: 65 | start loss: 4.710563659667969
